# ME324 · Lab 7 — Generating images (autoencoders & VAEs)

**Lecture 7 · "Generating images" · 2026-08-11**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsrobinson/me324/blob/main/labs/lab-07-autoencoders-vae.ipynb)

**How this notebook works.** Cells marked `# TODO` are for you to fill in — each one is
short and clearly marked. Worked answers for all of them are in the **Solutions** section
at the very bottom, so you will never be stuck for long. Have a real go first, then check.

**And please use an LLM.** ChatGPT, Claude, Gemini, DeepSeek — whichever you like.
*"In PyTorch, how do I …?"* is exactly the kind of question these tools are excellent at, and
looking things up this way is what every working researcher does. Two habits worth keeping:
ask for the **explanation** rather than just the line, and **run everything** it hands you.
The exam is closed-book, so what counts is that you can read the code back and say what it
does.

---

> **Turn the GPU on first.** In Colab: **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom. CPU also works, just more slowly.

Yesterday's CNN **recognised** images AND today's models will **create** them. By the end of the lab you
will have trained a network that writes new digits that exist in no dataset.

### Today's goals

1. Build and train a plain **autoencoder** — and see why it is *not* a generator
2. Fix it with a **variational autoencoder (VAE)**: a `(mu, logvar)` encoder, the
   **reparameterisation trick**, and a **reconstruction + KL** loss
3. **Generate**: sample `z ~ N(0, I)`, decode new digits, and interpolate between two
   digits in latent space

## ⏱️ Plan for today (~90 minutes)

This lab is built for a single 90-minute session, and it's **completely fine not to finish every cell in the room**.

- **Core — do these:** the autoencoder and the random-codes check, then the VAE — build, loss, train, sample.
- **Stretch / take-home — skip if short on time:** latent-space interpolation and the extensions.

_Most of the code is written for you; the `# TODO` cells are the parts you write. Worked answers are in the **Solutions** section at the bottom._

## Run me first

Imports, the course seed (1337), the device pick. Run it first.

In [ ]:
# If a package is missing (e.g. running locally), uncomment the next line:
# !pip install torch torchvision matplotlib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt

# Reproducibility: same seed -> same shuffles, same initial weights for everyone.
torch.manual_seed(1337)

# Generative models love a GPU. 'cuda' = an NVIDIA GPU; 'cpu' = your processor.
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch version:", torch.__version__)
print("Using device :", device)


## Section 0 · The data

Same as yesterday, we'll use `torchvision` downloads, and `DataLoader` to serve mini-batches.
We'll use the original **MNIST** dataset today, which contains 70,000 grayscale **handwritten digits** (28 x 28).


Two details matter:

* `ToTensor()` scales pixels to **[0, 1]**, and our decoder will end in a **sigmoid** —
  same range, so the network can match the data exactly
* We **flatten** each image to **784 numbers** (`28 * 28`), as today's models are plain
  fully-connected nets, and they want vectors (you could think about how you might incorporate convolutions into the general setup we will implement today as a separate, bonus exercise)

In [ ]:
# Same recipe as yesterday: download, convert to tensors, serve in batches.
train_data = datasets.MNIST(root="data", train=True,  download=True, transform=ToTensor())
test_data  = datasets.MNIST(root="data", train=False, download=True, transform=ToTensor())

BATCH_SIZE = 128
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=256,        shuffle=False)

print("training images:", len(train_data), "| test images:", len(test_data))
img, label = train_data[0]
print("one image tensor shape:", tuple(img.shape), "-> flattened to", 28 * 28, "numbers")
print("pixel value range:", float(img.min()), "to", float(img.max()))


### A plotting helper

Turns flat 784-vectors (or 28x28 images) into a grid of pictures. We will reuse this throughout:

In [ ]:
def show_images(images, title="", n=16, ncols=8):
    """Show up to n images. `images` can be (N, 784) flat vectors or (N, 1, 28, 28)."""
    images = images.detach().cpu().view(-1, 28, 28)   # collapse to (N, 28, 28)
    n = min(n, images.shape[0])
    nrows = (n + ncols - 1) // ncols
    plt.figure(figsize=(ncols, nrows + 0.5))
    for i in range(n):
        plt.subplot(nrows, ncols, i + 1)
        plt.imshow(images[i], cmap="gray")
        plt.axis("off")
    if title:
        plt.suptitle(title)
    plt.tight_layout()
    plt.show()

# Sanity check: look at a few real training digits.
example_imgs, example_labels = next(iter(train_loader))
show_images(example_imgs, title="A few real MNIST digits")


### One small config cell

* `HIDDEN` — width of the hidden layer.
* `LATENT_DIM` — the **bottleneck**: each 784-pixel image is squeezed to 20 numbers. It's this gap that *forces* an efficient summary.

In [ ]:
HIDDEN     = 256    # width of the hidden layer
LATENT_DIM = 20     # size of the compressed code (the bottleneck)
LR         = 1e-3   # learning rate for Adam


## Section 1 · A plain autoencoder

An **autoencoder** is two networks glued together at a narrow bottleneck:

```
   x (784)  ->  ENCODER  ->  z (20)  ->  DECODER  ->  x-hat (784)
   image        squeeze      code        expand       reconstruction
```

Notice this is "self-supervised" learning: the input is **its own target** (Lecture 7). To push 784 numbers through a
20-number bottleneck and still rebuild the image, the network is *forced* to learn the
structure of digits. We will measure closeness using the MSE,
$\mathcal{L}_{\text{recon}} = \sum_j (x_j - \hat{x}_j)^2$.

**Your first build.** The encoder is given; the decoder is its mirror image, plus a
final **sigmoid** so pixels land back in [0, 1]. Build it, then wire up `forward`.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, hidden=HIDDEN):
        super().__init__()
        # Encoder: 784 -> hidden -> latent_dim   (the squeeze)
        self.encoder = nn.Sequential(
            nn.Linear(784, hidden), nn.ReLU(),
            nn.Linear(hidden, latent_dim),
        )
        # TODO 1: build self.decoder — the encoder's mirror image (the expand):
        #   Linear(latent_dim -> hidden) -> ReLU -> Linear(hidden -> 784) -> Sigmoid
        self.decoder = None   # <-- replace None

    def forward(self, x):
        # TODO 2: squeeze x to a code z, expand z back to xhat, return (xhat, z).
        raise NotImplementedError("implement forward: squeeze to z, expand to xhat")

ae = Autoencoder().to(device)
print(ae)

### Training loop

The same loop from Labs 5 and 6: `zero_grad -> forward -> loss -> backward ->
step`. We just need to add a step to **flatten** batches to shape (batch, 784), and throw away the pre-packaged labels.

**Add the loss line.**

In [ ]:
def train_ae(model, loader, epochs=10, lr=LR):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        running = 0.0
        for images, _ in loader:                       # labels ignored: no labels needed!
            x = images.view(images.size(0), -1).to(device)   # flatten to (batch, 784)
            xhat, z = model(x)                                # forward
            loss = ...   # TODO: MSE of xhat vs x, summed over pixels, then / x.size(0)
            opt.zero_grad()
            loss.backward()
            opt.step()
            running += loss.item() * x.size(0)
        print(f"epoch {epoch+1:2d}/{epochs} | recon loss {running/len(loader.dataset):.2f}")
    return model

# Train it. (Bump epochs up later for better reconstructions.)
train_ae(ae, train_loader, epochs=10)

### Did it work? Originals vs reconstructions

Compare real test digits with their reconstructions. Expect clearly recognisable
digits, maybe just a little blurrier.

In [ ]:
ae.eval()
with torch.no_grad():
    imgs, _ = next(iter(test_loader))
    x = imgs.view(imgs.size(0), -1).to(device)
    xhat, _ = ae(x)

show_images(x,    title="Originals")
show_images(xhat, title="Autoencoder reconstructions")


### Can it generate? Predict first

Reconstruction works. So: invent a random code `z ~ N(0, I)` and run it through the
**decoder only** — does a brand-new digit come out? **Before you run the cell, predict
what the 16 images will look like, and why.** Think about *which* codes this decoder
has actually seen during training.

In [ ]:
# TODO: before you run this cell, write your prediction in the comment at the bottom.
ae.eval()
with torch.no_grad():
    # TODO: draw 16 random codes from N(0, I) — shape (16, LATENT_DIM) — on `device`.
    random_codes = None   # <-- replace None

if random_codes is None:
    print("Fill in the TODO above, then run this cell again.")
else:
    with torch.no_grad():
        images_out = ae.decoder(random_codes)   # decoder ONLY — the encoder never sees them
    show_images(images_out, title="Decoding random codes from a plain AE")


## Section 2 · The fix — a Variational Autoencoder (VAE)

The VAE *engineers* the latent space into something we can sample from. It changes
**three things** and nothing else (Lecture 7):

**Change 1 — the encoder outputs a *distribution*, not a point:** a mean `mu` and a
spread. For numerical stability we predict the **log of the variance** (`logvar`
$= \log \sigma^2$) and recover $\sigma = \exp(\tfrac12\,\text{logvar})$.

**Change 2 — sample the code (the reparameterisation trick).** Each forward pass draws

$$ z = \mu + \sigma \odot \epsilon, \qquad \epsilon \sim \mathcal{N}(0, I). $$

"Draw a random sample" is not differentiable — pulling the randomness into `eps`, an
*input* rather than a parameter, lets gradients flow through `mu` and `sigma` — that is
what makes the VAE trainable.

**Change 3 — a KL term in the loss** pulls every `(mu, sigma)` towards the standard
Gaussian `N(0, I)`, so that after training *any* draw from `N(0, I)` is a plausible
code. Details in a moment.

### Building the VAE

Build the class from the spec in the comments: a shared trunk, two heads (`fc_mu`,
`fc_logvar`), the autoencoder's decoder, and `reparameterise` — Change 2 as one line of
code.

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, hidden=HIDDEN):
        super().__init__()
        # TODO 1: build the VAE pieces.
        #   self.encoder   : Sequential(Linear(784 -> hidden), ReLU)   # shared trunk
        #   self.fc_mu     : Linear(hidden -> latent_dim)              # mean head
        #   self.fc_logvar : Linear(hidden -> latent_dim)              # log-variance head
        #   self.decoder   : Linear(latent_dim -> hidden) -> ReLU
        #                    -> Linear(hidden -> 784) -> Sigmoid
        raise NotImplementedError("build the VAE layers")

    def encode(self, x):
        # TODO 2: run the shared trunk, then return (mu, logvar) from the two heads.
        raise NotImplementedError("implement encode -> (mu, logvar)")

    def reparameterise(self, mu, logvar):
        # TODO 3: the reparameterisation trick.
        #   sigma = exp(0.5 * logvar); eps is a fresh draw from N(0, I) the same
        #   shape as sigma; combine them with mu as in Change 2. PyTorch has a
        #   one-call way to draw eps -- worth looking up.
        raise NotImplementedError("implement the reparameterisation trick")

    def decode(self, z):
        # TODO 4: expand a code z back into a 784-pixel image.
        raise NotImplementedError("implement decode")

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterise(mu, logvar)
        xhat = self.decode(z)
        return xhat, mu, logvar, z

vae = VAE().to(device)
print(vae)

### The VAE loss = reconstruction + KL

$$ \mathcal{L}_{\text{VAE}} = \underbrace{\mathcal{L}_{\text{recon}}(x, \hat{x})}_{\text{decoder does its job}} \;+\; \beta\cdot\underbrace{\text{KL}\big(\mathcal{N}(\mu,\sigma^2)\,\Vert\,\mathcal{N}(0, I)\big)}_{\text{latent stays a unit Gaussian}} $$

Reconstruction is the same MSE as before. The KL term has the lecture's closed form,
summed over the latent dimensions:

$$ \text{KL} = \tfrac{1}{2}\sum_{j}\big(\mu_j^2 + \sigma_j^2 - 1 - \ln \sigma_j^2\big). $$

In code we hold `logvar = ln(sigma^2)` directly, so the sum becomes
`-0.5 * sum(1 + logvar - mu.pow(2) - logvar.exp())`. `beta` is the lecture's $\beta$;
we use 1 — larger gives a smoother latent but blurrier images.

> **Quick check against the lecture.** One dimension, `mu = 0.5`, `sigma = 0.3`:
> $\tfrac12(0.25 + 0.09 - 1 + 2.408) \approx 0.87$ — the value worked out in the slides.

In [ ]:
BETA = 1.0   # weight on the KL term

def vae_loss(xhat, x, mu, logvar, beta=BETA):
    # TODO: implement the loss and return (recon + beta * kld, recon, kld), where
    #   recon : MSE summed over pixels, / x.size(0)  — exactly as in train_ae
    #   kld   : the closed form above, also / x.size(0) to average over the batch
    raise NotImplementedError("implement the VAE loss (reconstruction + KL)")

### Train the VAE

Same five-step rhythm — the lines that changed are yours. `vae_loss` returns all three
numbers, so you can watch reconstruction and KL trade off.

In [ ]:
def train_vae(model, loader, epochs=20, lr=LR):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        tot = tot_recon = tot_kld = 0.0
        for images, _ in loader:
            x = images.view(images.size(0), -1).to(device)      # flatten
            xhat, mu, logvar, z = ____                           # TODO 1: forward pass
            loss, recon, kld = ____                              # TODO 2: vae_loss(...)
            opt.zero_grad()
            ____                                                 # TODO 3: backpropagate
            opt.step()
            n = x.size(0)
            tot += loss.item() * n; tot_recon += recon.item() * n; tot_kld += kld.item() * n
        N = len(loader.dataset)
        print(f"epoch {epoch+1:2d}/{epochs} | total {tot/N:7.2f} | recon {tot_recon/N:7.2f} | KL {tot_kld/N:6.2f}")
    return model

train_vae(vae, train_loader, epochs=20)

### VAE reconstructions

Sanity check first: expect these a touch **blurrier** than the plain AE's — the KL term
trading a little sharpness for a latent we can sample. The payoff comes next.

In [ ]:
vae.eval()
with torch.no_grad():
    imgs, _ = next(iter(test_loader))
    x = imgs.view(imgs.size(0), -1).to(device)
    xhat, mu, logvar, z = vae(x)

show_images(x,    title="Originals")
show_images(xhat, title="VAE reconstructions")


## Section 3 · Generate new digits

This is the moment. The KL term pushed the latent towards `N(0, I)`, so the move that
produced junk from the AE should now work: sample 64 codes, decode. These digits are
copies of nothing in the dataset.

In [ ]:
vae.eval()
with torch.no_grad():
    # TODO 1: sample 64 codes z from N(0, I), on `device` — same move as the AE cell.
    # TODO 2: decode them into images (decoder only; the encoder plays no part).
    z          = None   # <-- replace None
    new_digits = None   # <-- replace None

if new_digits is None:
    print("Fill in the TODOs above, then run this cell again.")
else:
    show_images(new_digits, title="Brand-new digits sampled from N(0, I)", n=64, ncols=8)

### Interpolating in latent space

The cleanest evidence the latent space is *meaningful* — not a lookup table — is
**interpolation**: encode two digits to their mean codes, then decode points along the
line between them:

$$ z(t) = (1-t)\,z_1 + t\,z_2, \qquad t \in [0, 1]. $$

Every in-between code should decode to a plausible digit, so one digit **morphs
smoothly** into the other. Your job is the $z(t)$ line — `ts` is a column of 10 values
of $t$; broadcasting against `z1` and `z2` does the rest.

In [ ]:
vae.eval()
with torch.no_grad():
    imgs, labels = next(iter(test_loader))
    x = imgs.view(imgs.size(0), -1).to(device)
    mu, logvar = vae.encode(x)            # use the means as the two endpoints

    z1 = mu[0:1]                          # first image's code
    z2 = mu[1:2]                          # second image's code

    steps = 10
    ts = torch.linspace(0, 1, steps).view(-1, 1).to(device)
    # TODO: the straight line from z1 to z2 — the z(t) formula, applied to all of `ts`.
    z_path = None   # <-- replace None

if z_path is None:
    print("Fill in the TODO above, then run this cell again.")
else:
    with torch.no_grad():
        morph = vae.decode(z_path)
    print(f"morphing from a '{labels[0].item()}' to a '{labels[1].item()}'")
    show_images(morph, title="Interpolation in latent space", n=steps, ncols=steps)

### Why social scientists care: MIDAS

**MIDAS** (Lall & Robinson, 2022) uses a *denoising autoencoder* to impute **missing
values** in survey data: trained to reconstruct deliberately-corrupted observations, it
learns the joint structure of the variables and fills in the gaps. The technique is often more
accurate than MICE, and a lot faster at scale. It uses the same pieces you used today,
pointed at a substantive problem (Lectures 1 and 7).

## Recap

* A **plain autoencoder** compresses and reconstructs digits with MSE loss — but its
  latent space has *holes*, so random codes decode to junk.
* A **VAE** adds three things: a `(mu, logvar)` encoder, the
  **reparameterisation trick** `z = mu + sigma * eps`, and a **reconstruction + KL** loss.
* **Generation**: sample `z ~ N(0, I)`, decode brand-new digits, morph one digit into
  another.

The big shift this week: from **predictive** models (yesterday's CNN said *what* an
image is) to **generative** models (today's VAE *makes* new images).

### Extensions (optional)

* Set `LATENT_DIM = 2` and retrain. Reconstructions worsen — why? — but now you can
  scatter-plot `mu` for the test set, coloured by digit.
* Try `BETA = 0.1` and `4.0`: watch the recon/KL columns, then re-run the sampling cell.
* Bump the epochs on either model and see how much sharper the reconstructions get.

### Next time — Lab 8: text

Tomorrow we leave images behind: a character-level pipeline on tiny-shakespeare and our
first language model — the start of a four-lab arc that ends at a working **GPT**.

## Solutions

Worked answers for every `# TODO` above. Have a genuine go first — including asking a
model, which will often get you there before this section does.

To pick up where you left off: copy the answer into the matching `# TODO` cell above and
re-run from there, so the rest of the notebook uses your version.

**Solution — the autoencoder**

The decoder runs the encoder's shape story in reverse, plus the sigmoid. A chatbot may
offer separate layers with the sigmoid applied inside `forward` — identical maths.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, hidden=HIDDEN):
        super().__init__()
        # Encoder: 784 -> hidden -> latent_dim   (the squeeze)
        self.encoder = nn.Sequential(
            nn.Linear(784, hidden), nn.ReLU(),
            nn.Linear(hidden, latent_dim),
        )
        # Decoder: latent_dim -> hidden -> 784    (the expand)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, 784), nn.Sigmoid(),   # sigmoid -> pixels in [0, 1]
        )

    def forward(self, x):
        z = self.encoder(x)        # squeeze to the code
        xhat = self.decoder(z)     # expand back to an image
        return xhat, z

ae = Autoencoder().to(device)
print(ae)


**Solution — the reconstruction loss in `train_ae`**

`reduction="sum"` then `/ x.size(0)`: sum over pixels, average over the batch. A
chatbot's first offer, plain `F.mse_loss(xhat, x)`, averages over the 784 pixels too —
it still trains, but the number is 784x smaller and would sit on the wrong scale next
to the KL term in Section 2.

In [ ]:
def train_ae(model, loader, epochs=10, lr=LR):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        running = 0.0
        for images, _ in loader:                       # labels ignored: no labels needed!
            x = images.view(images.size(0), -1).to(device)   # flatten to (batch, 784)
            xhat, z = model(x)                                # 2. forward
            loss = F.mse_loss(xhat, x, reduction="sum") / x.size(0)   # 3. loss (MSE per image)
            opt.zero_grad()                                  # 5a. clear old grads
            loss.backward()                                  # 4. backward
            opt.step()                                       # 5b. update weights
            running += loss.item() * x.size(0)
        print(f"epoch {epoch+1:2d}/{epochs} | recon loss {running/len(loader.dataset):.2f}")
    return model

# Train it. (Bump epochs up later for sharper reconstructions.)
train_ae(ae, train_loader, epochs=10)


**Solution — decoding random codes from the AE**

You should have seen smeary blobs — nothing digit-like. The decoder has only ever seen
codes the *encoder produced from real digits*, and those live in a few lumpy clusters
with big empty gaps. A random `z` almost always lands in a gap the decoder was never
trained on, so out comes nonsense. The autoencoder *compresses*; it does not give us a
space we can *sample* from. That is the problem the VAE fixes.

In [ ]:
ae.eval()
with torch.no_grad():
    random_codes = torch.randn(16, LATENT_DIM).to(device)   # z ~ N(0, I)
    images_out = ae.decoder(random_codes)                   # decoder ONLY

show_images(images_out, title="Decoding random codes from a plain AE")

**Solution — the VAE class**

`torch.randn_like(std)` draws `eps` with the right shape and device in one call. Note
`z = mu + std * eps` is ordinary tensor arithmetic — the randomness sits in `eps`, so
gradients flow through `mu` and `std`. That is the whole trick.

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, hidden=HIDDEN):
        super().__init__()
        self.encoder   = nn.Sequential(nn.Linear(784, hidden), nn.ReLU())
        self.fc_mu     = nn.Linear(hidden, latent_dim)
        self.fc_logvar = nn.Linear(hidden, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, 784), nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterise(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + std * eps          # the reparameterisation trick
        return z

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterise(mu, logvar)
        xhat = self.decode(z)
        return xhat, mu, logvar, z

vae = VAE().to(device)
print(vae)

**Solution — the VAE loss**

The KL line is the closed form rearranged; `mu.pow(2)` and `logvar.exp()` are $\mu^2$
and $\sigma^2$. Both terms are divided by the batch size, so they stay comparable.

In [ ]:
BETA = 1.0

def vae_loss(xhat, x, mu, logvar, beta=BETA):
    recon = F.mse_loss(xhat, x, reduction="sum") / x.size(0)
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return recon + beta * kld, recon, kld

**Solution — the VAE training loop**

Only the forward and loss lines differ from `train_ae`; the five-step rhythm is
untouched.

In [ ]:
def train_vae(model, loader, epochs=20, lr=LR):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        tot = tot_recon = tot_kld = 0.0
        for images, _ in loader:
            x = images.view(images.size(0), -1).to(device)      # flatten
            xhat, mu, logvar, z = model(x)                       # forward
            loss, recon, kld = vae_loss(xhat, x, mu, logvar)    # loss = recon + KL
            opt.zero_grad()
            loss.backward()
            opt.step()
            n = x.size(0)
            tot += loss.item() * n; tot_recon += recon.item() * n; tot_kld += kld.item() * n
        N = len(loader.dataset)
        print(f"epoch {epoch+1:2d}/{epochs} | total {tot/N:7.2f} | recon {tot_recon/N:7.2f} | KL {tot_kld/N:6.2f}")
    return model

train_vae(vae, train_loader, epochs=20)


**Solution — sampling new digits**

Line for line the move that failed for the plain AE — the difference is entirely in the
latent space the KL term shaped.

In [ ]:
vae.eval()
with torch.no_grad():
    z = torch.randn(64, LATENT_DIM).to(device)   # sample from N(0, I)
    new_digits = vae.decode(z)                   # decoder turns codes into images

show_images(new_digits, title="Brand-new digits sampled from N(0, I)", n=64, ncols=8)


**Solution — latent interpolation**

Broadcasting does the work: `ts` is (10, 1), `z1` and `z2` are (1, 20), so
`(1 - ts) * z1 + ts * z2` is (10, 20) — one code per step along the line.

In [ ]:
vae.eval()
with torch.no_grad():
    imgs, labels = next(iter(test_loader))
    x = imgs.view(imgs.size(0), -1).to(device)
    mu, logvar = vae.encode(x)            # use the means as the two endpoints

    z1 = mu[0:1]                          # first image's code
    z2 = mu[1:2]                          # second image's code

    steps = 10
    ts = torch.linspace(0, 1, steps).view(-1, 1).to(device)
    z_path = (1 - ts) * z1 + ts * z2     # straight line from z1 to z2
    morph = vae.decode(z_path)

print(f"morphing from a '{labels[0].item()}' to a '{labels[1].item()}'")
show_images(morph, title="Interpolation in latent space", n=steps, ncols=steps)
